## ML_1019: Causal Self-Attention

**Difficulty**: Hard | **TorchCode**: #09

### Problem
Implement autoregressive (causal) attention where position `i` can only attend to positions `j ≤ i`. Mask future positions with `-inf` before softmax.

### Formula
$$\text{scores}_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}} + M_{ij}, \quad M_{ij} = \begin{cases} 0 & j \le i \\ -\infty & j > i \end{cases}$$

In [ ]:
import torch
import math

def causal_attention(Q, K, V):
    d_k = K.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    S = scores.size(-1)
    mask = torch.triu(torch.ones(S, S, device=scores.device, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return torch.bmm(weights, V)

In [ ]:
import torch

# --- Test 1: Output shape ---
out = causal_attention(torch.randn(2, 6, 16), torch.randn(2, 6, 16), torch.randn(2, 6, 16))
assert out.shape == (2, 6, 16)

# --- Test 2: Future tokens don't affect past ---
torch.manual_seed(0)
B, S, D = 1, 8, 16
Q = torch.randn(B, S, D); K = torch.randn(B, S, D); V = torch.randn(B, S, D)
out1 = causal_attention(Q, K, V)
K2, V2 = K.clone(), V.clone()
K2[:, 4:] = torch.randn(B, 4, D); V2[:, 4:] = torch.randn(B, 4, D)
out2 = causal_attention(Q, K2, V2)
assert torch.allclose(out1[:, :4], out2[:, :4], atol=1e-5)

# --- Test 3: First position only sees itself ---
torch.manual_seed(0)
Q = torch.randn(1, 4, 8); K = torch.randn(1, 4, 8); V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
assert torch.allclose(out[:, 0], V[:, 0], atol=1e-5)

# --- Test 4: Gradient flow ---
Q = torch.randn(2, 4, 8, requires_grad=True)
K = torch.randn(2, 4, 8, requires_grad=True)
V = torch.randn(2, 4, 8, requires_grad=True)
causal_attention(Q, K, V).sum().backward()
assert Q.grad is not None and K.grad is not None and V.grad is not None

print('All tests passed!')